# LSTM — Should You Buy This Stock?
**Dataset:** Fake 5-day stock price sequences (no download needed)
**Input:** 5 days of prices | **Output:** BUY (UP) or DONT BUY (DOWN)

> LSTM = Long Short-Term Memory. Like RNN but smarter — it uses gates to decide what to remember and what to forget.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

# ── Dataset ───────────────────────────────────────────────
# Each row: 5 consecutive daily prices -> Label 1=UP(BUY) 0=DOWN
raw = [
    # Rising trend -> BUY (1)
    ([100,102,105,107,110], 1), ([150,153,156,158,162], 1),
    ([200,205,210,213,218], 1), ([80, 83, 85, 88, 91],  1),
    ([120,122,125,128,132], 1), ([300,305,312,318,325], 1),
    ([50, 52, 54, 57, 60],  1), ([175,178,182,186,190], 1),
    ([90, 93, 95, 98,102],  1), ([250,255,260,265,272], 1),
    ([100,103,107,110,115], 1), ([60, 63, 65, 68, 72],  1),
    ([220,225,229,234,240], 1), ([130,133,136,140,145], 1),
    ([45, 47, 50, 53, 56],  1),
    # Falling trend -> DONT BUY (0)
    ([110,108,105,102, 99], 0), ([160,157,153,150,146], 0),
    ([218,213,208,204,200], 0), ([91, 88, 85, 82, 79],  0),
    ([132,129,125,121,118], 0), ([325,318,310,304,298], 0),
    ([60, 57, 54, 51, 48],  0), ([190,185,181,177,172], 0),
    ([102, 98, 95, 92, 88], 0), ([272,265,259,254,248], 0),
    ([115,111,107,103, 99], 0), ([72, 68, 65, 62, 58],  0),
    ([240,234,229,223,218], 0), ([145,140,136,132,127], 0),
    ([56, 52, 49, 46, 42],  0),
]

X = np.array([r[0] for r in raw], dtype='float32')
y = np.array([r[1] for r in raw])

# Normalize each sequence to 0-1
X = (X - X.min(axis=1, keepdims=True)) / \
    (X.max(axis=1, keepdims=True) - X.min(axis=1, keepdims=True) + 1e-8)
X = X.reshape(-1, 5, 1)   # (samples, 5 timesteps, 1 feature)

# ── Train / Test Split ─────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
# ── Build LSTM ────────────────────────────────────────────
model = Sequential([
    Input(shape=(5, 1)),             # 5 timesteps, 1 price feature
    LSTM(32),                        # remembers 5-day trends
    Dropout(0.2),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')   # 1=BUY, 0=DONT BUY
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────
history = model.fit(X_train, y_train, epochs=100, batch_size=5,
                    validation_data=(X_test, y_test), verbose=0)

print(f'Train Accuracy: {history.history["accuracy"][-1]*100:.1f}%')
print(f'Test  Accuracy: {history.history["val_accuracy"][-1]*100:.1f}%')

In [ ]:
# ── Accuracy Plot ─────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'],     label='Train')
plt.plot(history.history['val_accuracy'], label='Test', linestyle='--')
plt.title('LSTM — Training Accuracy over Epochs')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────
y_pred = (model.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
disp = ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                              display_labels=['DONT BUY','BUY'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — LSTM Stock Predictor')
plt.show()

In [ ]:
# ── Predict New Stocks ────────────────────────────────────
new_stocks = [
    {'prices': [100,103,106,109,113], 'hint': 'rising'},
    {'prices': [200,195,190,185,179], 'hint': 'falling'},
    {'prices': [50, 51, 52, 54, 56],  'hint': 'slow rise'},
]

print('─' * 50)
for s in new_stocks:
    arr = np.array(s['prices'], dtype='float32')
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    prob = model.predict(arr.reshape(1,5,1), verbose=0)[0][0]
    action = 'BUY ✅' if prob > 0.5 else 'DONT BUY ❌'
    print(f"{s['hint']:12s} {s['prices']} → {action} ({prob*100:.0f}%)")
print('─' * 50)